# 01 Factor Dashboard

Purpose: separate total return, benchmark beta, relative strength, volatility, and drawdown for AMD, TSM, NVDA, and GOOGL.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import pandas as pd

DB = '../data/duckdb/quant_learn.duckdb'
con = duckdb.connect(DB, read_only=True)

In [ ]:
latest = con.execute('select max(date) from factor_dashboard').fetchone()[0]
dashboard = con.execute('''
select *
from factor_dashboard
where date = ?
order by ticker
''', [latest]).fetchdf()
dashboard

In [ ]:
prices = con.execute('''
select date, ticker, adj_close, close
from prices
where ticker in ('AMD', 'TSM', 'NVDA', 'GOOGL', 'QQQ', 'SOXX', 'SMH')
order by date, ticker
''').fetchdf()
prices['date'] = pd.to_datetime(prices['date'])
prices['price'] = prices['adj_close'].fillna(prices['close'])
px = prices.pivot(index='date', columns='ticker', values='price')
(px / px.iloc[0]).plot(figsize=(12, 6), title='Growth of $1')
plt.grid(alpha=0.25)

Interpretation checklist:

- If a stock is up but relative return vs QQQ/SOXX is negative, the move is not stock-specific strength.
- If beta is above 1, market direction is a major driver.
- If return and volatility are both high, position sizing matters more than signal confidence.
- Use drawdown to understand the pain needed to hold through the cycle.